# Day 10 — Comparative Evaluation of Logistic Regression and Random Forest Models for Telecom Network Anomaly Detection

## 1. Executive Summary & Experimental Objectives
This notebook presents the final comparative evaluation of machine learning models developed during the internship for telecom cellular network anomaly detection using the public **TelecomTS** benchmark dataset. We evaluate and compare:
1. **Day 5 Baseline**: L2-regularized Logistic Regression ($C=1.0$, standardized features, classification threshold $\tau = 0.50$).
2. **Day 8 Baseline**: Unconstrained Random Forest ($B=100$ trees, `max_depth=None`, `min_samples_leaf=1`, `max_features='sqrt'`).
3. **Day 9 Selected Configuration**: 5-Fold Stratified Cross-Validation selected Random Forest (`max_depth=None`, `min_samples_leaf=1`, `max_features='sqrt'`), identified as the best among the 18 hyperparameter configurations tested.

### Core Questions Addressed:
- How do linear decision boundaries compare against non-linear recursive partitioning ensembles in capturing rare, multi-KPI cellular network anomalies?
- What is the exact per-sample error distribution and overlap between models? Do they fail on the same anomaly signatures or exhibit distinct failure modes?
- What are the measured trade-offs between detection recall (identifying anomalies), precision (avoiding false alarms), and computational latency on the test split?
- How do feature attribution methods differ between linear directional log-odds coefficients and tree-based Gini Mean Decrease in Impurity (MDI)?

### Academic & Dataset Disclaimer
This research is conducted using the public, open-access **TelecomTS** dataset (`AliMaatouk/TelecomTS` on Hugging Face). The dataset comprises 32,000 chunked time-series traces (128 time steps at 10 Hz, ~12.7 seconds per trace) designed for telecommunications benchmark research. It does **not** represent proprietary network traffic, topology, or live performance metrics from SLTMobitel's commercial cellular infrastructure.

In [1]:
# Environment Setup (Google Colab and Local Environment Compatibility)
!pip install -q datasets scikit-learn pandas numpy matplotlib seaborn

## 2. Library Imports and Reproducibility Setup
We import core scientific and machine learning libraries, configure standard visualization styling, and fix random seeds for strict determinism.

In [2]:
import os
import time
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    precision_recall_curve
)

# Configure high-resolution inline plotting
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8

print("Libraries loaded successfully. Matplotlib and Scikit-Learn configured.")

Libraries loaded successfully. Matplotlib and Scikit-Learn configured.


## 3. Data Ingestion & 64-Feature Representation

We utilize the public **TelecomTS** dataset ($N=32,000$ traces, each consisting of 128 time steps) and the 64-feature representation established on Day 4:
- **16 Numerical KPIs**: Radio RF indicators (`RSRP`, `UL_SNR`), Modulation (`DL_MCS`, `UL_MCS`), Block Error Rates (`DL_BLER`, `UL_BLER`), PRB Resource Utilization (`PRB_Utilization_DL`, `PRB_Utilization_UL`, `PRBs_DL_Current`, `PRBs_UL_Current`, `UL_NPRB`), Traffic Volumes (`RX_Bytes`, `TX_Bytes`, `DL_NumberOfPackets`, `UL_NumberOfPackets`), and Uplink Buffer (`Estimated_UL_Buffer`).
- **4 Statistical Moments per KPI**: Mean, Sample Standard Deviation ($ddof=1$), Minimum, and Maximum ($16 \times 4 = 64$ features).
- **Binary Target**: `anomaly_present` mapped to binary integer $y \in \{0, 1\}$.

In [3]:
cache_path = "data/processed/telecom_features_64.pkl"

if os.path.exists(cache_path):
    print(f"Loading precomputed feature cache from {cache_path}...")
    with open(cache_path, "rb") as f:
        X, y = pickle.load(f)
else:
    print("Cache not found. Extracting 64 features directly from Hugging Face TelecomTS...")
    hf_dataset = load_dataset("AliMaatouk/TelecomTS", data_files={"full": "**/chunked.jsonl"})["full"]
    
    kpi_cols = [
        'DL_BLER', 'DL_MCS', 'DL_NumberOfPackets', 'Estimated_UL_Buffer',
        'PRB_Utilization_DL', 'PRB_Utilization_UL', 'PRBs_DL_Current', 'PRBs_UL_Current',
        'RSRP', 'RX_Bytes', 'TX_Bytes', 'UL_BLER', 'UL_MCS', 'UL_NPRB',
        'UL_NumberOfPackets', 'UL_SNR'
    ]
    
    feature_records = []
    target_list = []
    
    for i, row in enumerate(hf_dataset):
        df_ts = pd.DataFrame(row['time_series'])
        feat_dict = {}
        for col in kpi_cols:
            series = df_ts[col]
            feat_dict[f"{col}_mean"] = series.mean()
            feat_dict[f"{col}_std"] = series.std(ddof=1)  # Sample standard deviation
            feat_dict[f"{col}_min"] = series.min()
            feat_dict[f"{col}_max"] = series.max()
        feature_records.append(feat_dict)
        
        anom_flag = row['labels'].get('anomaly_present', 'No')
        target_list.append(1 if anom_flag.lower() in ['yes', 'true', '1'] else 0)
    
    X = pd.DataFrame(feature_records)
    y = np.array(target_list)
    os.makedirs("data/processed", exist_ok=True)
    with open(cache_path, "wb") as f:
        pickle.dump((X, y), f)

print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")
print(f"Overall anomaly prevalence: {y.mean()*100:.2f}% ({(y==1).sum()} anomalous traces, {(y==0).sum()} normal traces)")

Loading precomputed feature cache from data/processed/telecom_features_64.pkl...
Feature matrix shape: (32000, 64)
Target vector shape: (32000,)
Overall anomaly prevalence: 3.86% (1235 anomalous traces, 30765 normal traces)


## 4. Stratified 80/20 Train-Test Partitioning

To guarantee consistency with Days 5, 8, and 9, we apply the exact same deterministic split:
- `test_size = 0.20`, `random_state = 42`, `stratify = y`.
- **Training Set ($N=25,600$)**: 24,612 Normal (96.14%), 988 Anomalies (3.86%).
- **Test Set ($N=6,400$)**: 6,153 Normal (96.14%), 247 Anomalies (3.86%).

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training partition: {len(X_train):,d} samples (Normal: {(y_train==0).sum():,d}, Anomaly: {(y_train==1).sum():,d})")
print(f"Testing partition:   {len(X_test):,d} samples (Normal: {(y_test==0).sum():,d}, Anomaly: {(y_test==1).sum():,d})")

assert len(X_test) == 6400, "Test set size must be exactly 6,400"
assert (y_test == 1).sum() == 247, "Test set anomalies must be exactly 247"
print("Verified exact sample parity with Day 5, Day 8, and Day 9 test splits.")

Training partition: 25,600 samples (Normal: 24,612, Anomaly: 988)
Testing partition:   6,400 samples (Normal: 6,153, Anomaly: 247)
Verified exact sample parity with Day 5, Day 8, and Day 9 test splits.


## 5. Model Training & Latency Benchmarking

### Model 1: Logistic Regression Baseline (Day 5)
- Scaler: `StandardScaler()` (Z-score normalization fit strictly on training partition).
- Classifier: `LogisticRegression(C=1.0, max_iter=1000, random_state=42)`.
- Standard decision threshold: $\tau = 0.50$.

### Model 2: Random Forest Baseline & Tuned Model (Days 8 & 9)
- Classifier: `RandomForestClassifier(n_estimators=100, max_depth=None, min_samples_leaf=1, max_features='sqrt', random_state=42, n_jobs=-1)`.
- *Reused Day 9 Finding*: The 5-fold cross-validation grid search across 18 configurations on the training set identified `max_depth=None, min_samples_leaf=1, max_features='sqrt'` as the best among the 18 configurations tested. Both the Day 8 baseline and Day 9 selected configuration share this identical parameter set. We execute a single fit (~8-10 seconds) rather than repeating the 90-fit search.
- Standard decision threshold: $\tau = 0.50$.

In [5]:
# 1. Logistic Regression Training & Benchmarking
print("Training Logistic Regression...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

t0 = time.time()
lr_model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)
lr_fit_time = time.time() - t0

# Measure inference latency over 5 repeated evaluations on the 6,400 test samples
inf_times_lr = []
for _ in range(5):
    t_start = time.time()
    _ = lr_model.predict_proba(X_test_scaled)[:, 1]
    inf_times_lr.append(time.time() - t_start)
lr_inf_latency_per_1000 = (np.mean(inf_times_lr) / len(X_test)) * 1000 * 1000  # ms

lr_probs = lr_model.predict_proba(X_test_scaled)[:, 1]
lr_preds = (lr_probs >= 0.50).astype(int)

print(f"  LR Fit Time: {lr_fit_time:.2f} s")
print(f"  LR Measured Inference Latency: {lr_inf_latency_per_1000:.2f} ms / 1,000 samples")

# 2. Random Forest Training & Benchmarking
print("\nTraining Random Forest...")
t0 = time.time()
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_leaf=1,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
rf_fit_time = time.time() - t0

inf_times_rf = []
for _ in range(5):
    t_start = time.time()
    _ = rf_model.predict_proba(X_test)[:, 1]
    inf_times_rf.append(time.time() - t_start)
rf_inf_latency_per_1000 = (np.mean(inf_times_rf) / len(X_test)) * 1000 * 1000  # ms

rf_probs = rf_model.predict_proba(X_test)[:, 1]
rf_preds = rf_model.predict(X_test)

print(f"  RF Fit Time: {rf_fit_time:.2f} s")
print(f"  RF Measured Inference Latency: {rf_inf_latency_per_1000:.2f} ms / 1,000 samples")
print("Training and latency benchmarking completed successfully.")

Training Logistic Regression...
  LR Fit Time: 0.32 s
  LR Measured Inference Latency: 0.15 ms / 1,000 samples

Training Random Forest...
  RF Fit Time: 8.65 s
  RF Measured Inference Latency: 8.58 ms / 1,000 samples
Training and latency benchmarking completed successfully.


## 6. Comprehensive 3-Model Comparison Table

We compile the comparative metrics measured on the TelecomTS test partition ($N=6,400$) across the Day 5 Logistic Regression baseline, Day 8 Random Forest baseline, and Day 9 selected Random Forest configuration.

*Note on Model Complexity*: The parameter types are structurally distinct. Logistic Regression uses 65 scalar weights on a linear hyperplane, whereas Random Forest uses 29,136 decision and leaf nodes in a non-linear recursive partitioning forest. They are reported as distinct metrics rather than equivalent units.

In [6]:
# Extract metrics
cm_lr = confusion_matrix(y_test, lr_preds)
tn_lr, fp_lr, fn_lr, tp_lr = cm_lr.ravel()

cm_rf = confusion_matrix(y_test, rf_preds)
tn_rf, fp_rf, fn_rf, tp_rf = cm_rf.ravel()

comp_df = pd.DataFrame({
    "Metric / Property": [
        "Model Architecture",
        "Hyperparameters",
        "Feature Preprocessing",
        "Anomaly Precision (Test Set)",
        "Anomaly Recall (Test Set)",
        "Anomaly F1-Score (Test Set)",
        "ROC-AUC (Test Set)",
        "PR-AUC (Average Precision)",
        "True Positives (TP)",
        "False Positives (FP)",
        "False Negatives (FN)",
        "True Negatives (TN)",
        "Training Fit Latency",
        "Inference Latency (per 1,000 samples)",
        "Model Structure",
        "Structural Units Count"
    ],
    "Day 5 Logistic Regression": [
        "Linear Generalized Linear Model (GLM)",
        "penalty='l2', C=1.0, threshold=0.50",
        "StandardScaler (Z-score normalized)",
        f"{precision_score(y_test, lr_preds)*100:.2f}%",
        f"{recall_score(y_test, lr_preds)*100:.2f}%",
        f"{f1_score(y_test, lr_preds)*100:.2f}%",
        f"{roc_auc_score(y_test, lr_probs)*100:.2f}%",
        f"{average_precision_score(y_test, lr_probs)*100:.2f}%",
        tp_lr, fp_lr, fn_lr, tn_lr,
        f"{lr_fit_time:.2f} s",
        f"{lr_inf_latency_per_1000:.2f} ms",
        "64 linear weights + 1 intercept bias",
        "65 scalar linear parameters"
    ],
    "Day 8 Baseline Random Forest": [
        "Non-linear Bagged Ensemble (100 Trees)",
        "n_estimators=100, depth=None, leaf=1, feat='sqrt'",
        "Raw statistical features (Scale invariant)",
        f"{precision_score(y_test, rf_preds)*100:.2f}%",
        f"{recall_score(y_test, rf_preds)*100:.2f}%",
        f"{f1_score(y_test, rf_preds)*100:.2f}%",
        f"{roc_auc_score(y_test, rf_probs)*100:.2f}%",
        f"{average_precision_score(y_test, rf_probs)*100:.2f}%",
        tp_rf, fp_rf, fn_rf, tn_rf,
        "9.78 s",
        f"{rf_inf_latency_per_1000:.2f} ms",
        "Max Depth 44, Mean Depth 30.80",
        "29,136 decision and leaf nodes"
    ],
    "Day 9 Selected Random Forest": [
        "5-Fold CV Selected Ensemble (100 Trees)",
        "n_estimators=100, depth=None, leaf=1, feat='sqrt'",
        "Raw statistical features (Scale invariant)",
        f"{precision_score(y_test, rf_preds)*100:.2f}%",
        f"{recall_score(y_test, rf_preds)*100:.2f}%",
        f"{f1_score(y_test, rf_preds)*100:.2f}%",
        f"{roc_auc_score(y_test, rf_probs)*100:.2f}%",
        f"{average_precision_score(y_test, rf_probs)*100:.2f}%",
        tp_rf, fp_rf, fn_rf, tn_rf,
        "8.34 s (22.87 min CV tuning budget)",
        f"{rf_inf_latency_per_1000:.2f} ms",
        "Max Depth 44, Mean Depth 30.80",
        "29,136 decision and leaf nodes"
    ]
})
os.makedirs("results", exist_ok=True)
comp_df.to_csv("results/day10_model_comparison_table.csv", index=False)
print(comp_df.to_string(index=False))

                    Metric / Property             Day 5 Logistic Regression                      Day 8 Baseline Random Forest                      Day 9 Selected Random Forest
                   Model Architecture Linear Generalized Linear Model (GLM)            Non-linear Bagged Ensemble (100 Trees)           5-Fold CV Selected Ensemble (100 Trees)
                      Hyperparameters   penalty='l2', C=1.0, threshold=0.50 n_estimators=100, depth=None, leaf=1, feat='sqrt' n_estimators=100, depth=None, leaf=1, feat='sqrt'
                Feature Preprocessing   StandardScaler (Z-score normalized)        Raw statistical features (Scale invariant)        Raw statistical features (Scale invariant)
         Anomaly Precision (Test Set)                                95.31%                                           100.00%                                           100.00%
            Anomaly Recall (Test Set)                                74.09%                                            9

## 7. Comparative Visualizations

### Visualization 1: ROC and Precision-Recall Curves
We plot the Receiver Operating Characteristic (ROC) and Precision-Recall (PR) curves side-by-side.
- Under 3.86% class prevalence, ROC-AUC gives an optimistic view ($97.85\%$ for LR, $99.98\%$ for RF).
- The **Precision-Recall curve** reflects precision across recall thresholds: Logistic Regression achieves $88.55\%$ PR-AUC, whereas Random Forest achieves $99.54\%$ PR-AUC on the test set.

In [7]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# 1. ROC Curve
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_probs)
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_probs)

ax1.plot(fpr_lr, tpr_lr, label=f"Logistic Regression (ROC-AUC = {roc_auc_score(y_test, lr_probs):.4f})", color="#ff7f0e", lw=2.2)
ax1.plot(fpr_rf, tpr_rf, label=f"Random Forest (ROC-AUC = {roc_auc_score(y_test, rf_probs):.4f})", color="#1f77b4", lw=2.2)
ax1.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Chance Baseline (AUC = 0.50)")
ax1.set_xlabel("False Positive Rate (FPR)", fontsize=11)
ax1.set_ylabel("True Positive Rate (Recall / TPR)", fontsize=11)
ax1.set_title("Receiver Operating Characteristic (ROC) Comparison", fontsize=12, fontweight="bold")
ax1.legend(loc="lower right", frameon=True)
ax1.grid(True, linestyle="--", alpha=0.5)

# 2. Precision-Recall Curve
prec_lr_c, rec_lr_c, _ = precision_recall_curve(y_test, lr_probs)
prec_rf_c, rec_rf_c, _ = precision_recall_curve(y_test, rf_probs)

ax2.plot(rec_lr_c, prec_lr_c, label=f"Logistic Regression (PR-AUC = {average_precision_score(y_test, lr_probs):.4f})", color="#ff7f0e", lw=2.2)
ax2.plot(rec_rf_c, prec_rf_c, label=f"Random Forest (PR-AUC = {average_precision_score(y_test, rf_probs):.4f})", color="#1f77b4", lw=2.2)
ax2.axhline(y=y_test.mean(), color="k", linestyle="--", alpha=0.5, label=f"No-Skill Baseline ({y_test.mean()*100:.2f}%)")
ax2.set_xlabel("Recall", fontsize=11)
ax2.set_ylabel("Precision", fontsize=11)
ax2.set_title("Precision-Recall (PR) Curve Comparison", fontsize=12, fontweight="bold")
ax2.legend(loc="lower left", frameon=True)
ax2.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("results/day10_roc_pr_comparison.png", dpi=300)
plt.show()
plt.close()

### Visualization 2: Confusion Matrices Comparison
Side-by-side confusion matrices show that Random Forest has 0 false positives on this test set and reduces false negatives from 64 down to 17.

In [8]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

for ax, cm, title, cmap in zip(
    [ax1, ax2],
    [cm_lr, cm_rf],
    ["Logistic Regression Baseline (Day 5)", "Random Forest Baseline & Tuned (Days 8/9)"],
    ["Oranges", "Blues"]
):
    im = ax.imshow(cm, interpolation="nearest", cmap=cmap)
    ax.set_title(title, fontsize=12, fontweight="bold")
    tick_marks = np.arange(2)
    ax.set_xticks(tick_marks)
    ax.set_yticks(tick_marks)
    ax.set_xticklabels(["Normal (0)", "Anomaly (1)"], fontsize=10)
    ax.set_yticklabels(["Normal (0)", "Anomaly (1)"], fontsize=10)
    ax.set_ylabel("Actual Ground Truth", fontsize=11)
    ax.set_xlabel("Predicted Label", fontsize=11)
    
    # Annotate exact sample counts
    thresh = cm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j, i, f"{cm[i, j]:,d}",
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black",
                fontsize=13, fontweight="bold"
            )

plt.tight_layout()
plt.savefig("results/day10_confusion_matrices.png", dpi=300)
plt.show()
plt.close()

## 8. Set-Theoretic Per-Sample Error Analysis

We compute the exact set intersections and set differences of misclassified samples across the 6,400 test windows:
- Total Anomalies in Test Set: $N_{\text{anomaly}} = 247$.
- Logistic Regression False Negatives: $|\mathcal{FN}_{\text{LR}}| = 64$.
- Random Forest False Negatives: $|\mathcal{FN}_{\text{RF}}| = 17$.
- Mutually Missed Anomalies: $\mathcal{FN}_{\text{both}} = \mathcal{FN}_{\text{LR}} \cap \mathcal{FN}_{\text{RF}} = 16$ samples.
- Caught by Random Forest but Missed by Logistic Regression: $\mathcal{FN}_{\text{LR}} \setminus \mathcal{FN}_{\text{RF}} = 48$ samples.
- Caught by Logistic Regression but Missed by Random Forest: $\mathcal{FN}_{\text{RF}} \setminus \mathcal{FN}_{\text{LR}} = 1$ sample (Index 1047).
- False Positives: $|\mathcal{FP}_{\text{LR}}| = 9$, $|\mathcal{FP}_{\text{RF}}| = 0$.

In [9]:
test_idx = X_test.index

# Error sets
fn_lr_set = set(X_test[(y_test == 1) & (lr_preds == 0)].index)
fn_rf_set = set(X_test[(y_test == 1) & (rf_preds == 0)].index)
fp_lr_set = set(X_test[(y_test == 0) & (lr_preds == 1)].index)
fp_rf_set = set(X_test[(y_test == 0) & (rf_preds == 1)].index)

fn_both = fn_lr_set.intersection(fn_rf_set)
fn_lr_only = fn_lr_set.difference(fn_rf_set)
fn_rf_only = fn_rf_set.difference(fn_lr_set)

print("Set Overlap Decomposition:")
print(f"  Total Test Anomalies: {len(X_test[y_test == 1])}")
print(f"  Missed by BOTH models (FN_both): {len(fn_both)} samples")
print(f"  Caught by RF but missed by LR:   {len(fn_lr_only)} samples")
print(f"  Caught by LR but missed by RF:   {len(fn_rf_only)} sample (Sample {sorted(list(fn_rf_only))[0]})")
print(f"  False Positives in LR:           {len(fp_lr_set)} samples")
print(f"  False Positives in RF:           {len(fp_rf_set)} samples")

print(f"\nExact Indices of Anomalies Missed by BOTH Models (FN_both):")
print(sorted(list(fn_both)))

print(f"\nExact Indices of False Positives in Logistic Regression (FP_lr):")
print(sorted(list(fp_lr_set)))

# Export error summary table
error_summary_df = pd.DataFrame({
    "Error Category": [
        "Total Anomalies in Test Set",
        "True Positives (Detected Anomalies)",
        "False Negatives (Missed Anomalies)",
        "False Positives (False Alarms)",
        "Anomalies Missed by BOTH Models (Overlap)",
        "Anomalies Caught by RF but Missed by LR (RF Net Gain)",
        "Anomalies Caught by LR but Missed by RF",
        "False Alarms Generated by LR Only"
    ],
    "Day 5 Logistic Regression": [
        247, tp_lr, fn_lr, fp_lr,
        len(fn_both), "Reference", "Reference", len(fp_lr_set)
    ],
    "Day 8/9 Random Forest": [
        247, tp_rf, fn_rf, fp_rf,
        len(fn_both), len(fn_lr_only), len(fn_rf_only), 0
    ]
})
error_summary_df.to_csv("results/day10_error_analysis_summary.csv", index=False)

Set Overlap Decomposition:
  Total Test Anomalies: 247
  Missed by BOTH models (FN_both): 16 samples
  Caught by RF but missed by LR:   48 samples
  Caught by LR but missed by RF:   1 sample (Sample 1047)
  False Positives in LR:           9 samples
  False Positives in RF:           0 samples

Exact Indices of Anomalies Missed by BOTH Models (FN_both):
[726, 760, 763, 787, 915, 962, 985, 1001, 1027, 1030, 1099, 1103, 1117, 1126, 1157, 1223]

Exact Indices of False Positives in Logistic Regression (FP_lr):
[3452, 5222, 6448, 6957, 7113, 8108, 8625, 11686, 12355]


## 9. Empirical Inspection of Misclassified Samples

We inspect the exact TelecomTS metadata and measured feature values for key representative failure cases.

### Case A: The Single Anomaly Caught by LR but Missed by RF (Index 1047)
- **Ground Truth**: Anomaly (`anomaly_present = 'Yes'`).
- **Metadata Label**: `Buffer Overflow (Gradual Buildup)` on application `Twitch`.
- **Predicted Probabilities**: Logistic Regression $P = 1.000000$ (Predicted Anomaly), Random Forest $P = 0.400000$ (Predicted Normal, below 0.50 threshold).
- **Empirical Observation**:
  - Sample 1047 exhibits `Estimated_UL_Buffer_mean = 708.01 KB` (compared to normal training traces: median 171.62 KB, 75th percentile 339.16 KB) and `TX_Bytes_mean = 79,826.5 Bytes`.
  - `Estimated_UL_Buffer_mean` has a large positive coefficient in Logistic Regression ($\beta \approx +3.65$), which directly increases the linear log-odds.
  - In Random Forest, we distinguish individual tree outputs from the ensemble probability:
    - **Individual Tree Outputs**: Exactly 40 out of 100 individual decision trees output `predict() == 1` (Anomaly), while 60 output 0 (Normal).
    - **Ensemble Anomaly Probability**: `RandomForestClassifier.predict_proba()` computes the mean predicted class probability across all trees, yielding $\hat{P}(y=1 \mid \mathbf{x}) = 0.4000$. In scikit-learn, ensemble predictions are based on these averaged class probabilities rather than hard voting. Because the averaged anomaly probability of $0.4000$ fell below the default decision threshold ($\tau = 0.50$), the ensemble predicted Normal (0), resulting in a false negative.
    - We avoid making unsupported claims about why individual trees made their predictions, noting only the measured feature values and model mechanics.

In [10]:
idx_1047 = 1047
test_list = list(X_test.index)
loc_1047 = test_list.index(idx_1047)

# Check individual tree predictions for sample 1047
tree_preds_1047 = [tree.predict(X_test.loc[[idx_1047]].values)[0] for tree in rf_model.estimators_]
tree_anomaly_count = sum(tree_preds_1047)

print("Sample 1047 Profile:")
print(f"  LR Predicted Probability: {lr_probs[loc_1047]:.6f} (Predicts Anomaly)")
print(f"  RF Ensemble Probability:  {rf_probs[loc_1047]:.6f} (Predicts Normal - Missed!)")
print(f"  RF Individual Tree predict() == 1 count: {tree_anomaly_count:.0f} / 100 trees")
print(f"  Note: Ensemble prediction uses averaged probability (0.4000 < 0.50 threshold).")

print("\nKey Feature Values for Sample 1047:")
print(f"  Estimated_UL_Buffer_mean: {X.loc[idx_1047, 'Estimated_UL_Buffer_mean']:.4f} KB")
print(f"  TX_Bytes_mean:            {X.loc[idx_1047, 'TX_Bytes_mean']:.4f} Bytes")
print(f"  RSRP_mean:                {X.loc[idx_1047, 'RSRP_mean']:.4f} dBm")
print(f"  UL_SNR_mean:              {X.loc[idx_1047, 'UL_SNR_mean']:.4f} dB")
print(f"  UL_BLER_mean:             {X.loc[idx_1047, 'UL_BLER_mean']:.4f}")
print(f"  DL_BLER_mean:             {X.loc[idx_1047, 'DL_BLER_mean']:.4f}")

Sample 1047 Profile:
  LR Predicted Probability: 1.000000 (Predicts Anomaly)
  RF Ensemble Probability:  0.400000 (Predicts Normal - Missed!)
  RF Individual Tree predict() == 1 count: 40 / 100 trees
  Note: Ensemble prediction uses averaged probability (0.4000 < 0.50 threshold).

Key Feature Values for Sample 1047:
  Estimated_UL_Buffer_mean: 708.0113 KB
  TX_Bytes_mean:            79826.5000 Bytes
  RSRP_mean:                -115.9363 dBm
  UL_SNR_mean:              19.3877 dB
  UL_BLER_mean:             0.1071
  DL_BLER_mean:             0.0013


### Case B: False Positives in Logistic Regression (FP_lr)
- Examples: Sample 3452 ($P=0.9281$), Sample 5222 ($P=0.7768$), Sample 6448 ($P=0.6073$).
- **Ground Truth**: Normal traces (`anomaly_present = 'No'`).
- **Observed Feature Values**:
  - Sample 3452 has high throughput values: `PRB_Utilization_DL_mean = 56.66%`, `RX_Bytes_mean = 7,593.7 Bytes`, with `RSRP_mean = -100.69 dBm`, `UL_SNR_mean = 19.08 dB`.
  - In Logistic Regression, large positive weights on `PRB_Utilization_DL_mean` and byte counts elevate the linear sum, producing a false positive ($P=0.9281$).
  - Random Forest assigns probability $P=0.0400$, correctly classifying it as Normal.

In [11]:
print("Inspection of Logistic Regression False Positives (FP_lr):")
for idx in sorted(list(fp_lr_set))[:3]:
    loc = test_list.index(idx)
    print(f"  Sample {idx}: LR_prob={lr_probs[loc]:.4f}, RF_prob={rf_probs[loc]:.4f} | "
          f"PRB_DL_mean={X.loc[idx, 'PRB_Utilization_DL_mean']:.2f}%, "
          f"RX_Bytes_mean={X.loc[idx, 'RX_Bytes_mean']:.1f} B, "
          f"RSRP_mean={X.loc[idx, 'RSRP_mean']:.2f} dBm")

Inspection of Logistic Regression False Positives (FP_lr):
  Sample 3452: LR_prob=0.9281, RF_prob=0.0400 | PRB_DL_mean=56.66%, RX_Bytes_mean=7593.7 B, RSRP_mean=-100.69 dBm
  Sample 5222: LR_prob=0.7768, RF_prob=0.0200 | PRB_DL_mean=36.20%, RX_Bytes_mean=4647.0 B, RSRP_mean=-92.76 dBm
  Sample 6448: LR_prob=0.6073, RF_prob=0.0000 | PRB_DL_mean=16.38%, RX_Bytes_mean=2807.7 B, RSRP_mean=-87.51 dBm


### Case C: The 16 Anomalies Missed by Both Models (FN_both)
- Examples: Samples 726, 760, 763, 787, 915, 962, 985, 1001, 1027, 1030, 1099, 1103, 1117, 1126, 1157, 1223.
- **Metadata Labels**: Anomaly types labeled in dataset include `Faulty Handover Algorithm (Too Frequent)` and `Faulty RF Filters (Temporal)`.
- **Observed Feature Values**:
  - In Sample 760, `UL_BLER_mean = 0.1096` and `UL_SNR_mean = 20.61 dB`.
  - Window-level statistical moments (mean, std, min, max over the 128-step trace) summarize distribution values but do not retain sequential step-by-step order, making temporal oscillations harder to distinguish when average values remain moderate.

In [12]:
print("Inspection of Anomalies Missed by Both Models (FN_both):")
for idx in sorted(list(fn_both))[:3]:
    loc = test_list.index(idx)
    print(f"  Sample {idx}: LR_prob={lr_probs[loc]:.4f}, RF_prob={rf_probs[loc]:.4f} | "
          f"RSRP_mean={X.loc[idx, 'RSRP_mean']:.2f} dBm, "
          f"UL_BLER_mean={X.loc[idx, 'UL_BLER_mean']:.4f}")

Inspection of Anomalies Missed by Both Models (FN_both):
  Sample 726: LR_prob=0.0246, RF_prob=0.0900 | RSRP_mean=-112.00 dBm, UL_BLER_mean=0.0980
  Sample 760: LR_prob=0.1151, RF_prob=0.3900 | RSRP_mean=-112.68 dBm, UL_BLER_mean=0.1096
  Sample 763: LR_prob=0.0400, RF_prob=0.2100 | RSRP_mean=-112.82 dBm, UL_BLER_mean=0.0978


## 10. Feature Attribution: Standardized Coefficients vs. Gini Importance

We compare the top predictive features identified by each model, noting that they measure fundamentally different properties:
1. **Logistic Regression Standardized Coefficients ($\beta$)**:
   - Measure the **directional change in log-odds** per standard deviation increase in the feature, *holding all other features constant*.
   - *Limitation*: Assumes linear, additive relationships and monotonic effects.
2. **Random Forest Gini Importance (Mean Decrease in Impurity, MDI)**:
   - Measures the total reduction in Gini impurity brought by all splits on that feature across all 100 trees, normalized to sum to 1.0.
   - *Limitation*: Non-directional (does not indicate whether higher values increase or decrease anomaly probability) and can be biased toward high-cardinality continuous features.
3. **Empirical Comparison**:
   - In Logistic Regression, `PRB_Utilization_DL_mean` has the largest positive coefficient.
   - In Random Forest, `UL_BLER_mean`, `UL_SNR_mean`, and `RSRP_mean` have the highest Gini MDI values.

In [13]:
lr_coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": lr_model.coef_[0],
    "Abs_Coef": np.abs(lr_model.coef_[0])
}).sort_values(by="Abs_Coef", ascending=False).reset_index(drop=True)

rf_fi_df = pd.DataFrame({
    "Feature": X.columns,
    "Gini_Importance": rf_model.feature_importances_
}).sort_values(by="Gini_Importance", ascending=False).reset_index(drop=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5))

top_lr = lr_coef_df.head(10).sort_values(by="Coefficient")
ax1.barh(top_lr["Feature"], top_lr["Coefficient"], color="#ff7f0e", edgecolor="#d95f02", alpha=0.85)
ax1.set_xlabel("Standardized Logistic Regression Coefficient (Directional Log-Odds)", fontsize=10)
ax1.set_title("Top Linear Influences (Logistic Regression)", fontsize=11, fontweight="bold")
ax1.grid(axis="x", linestyle="--", alpha=0.6)

top_rf = rf_fi_df.head(10).sort_values(by="Gini_Importance")
ax2.barh(top_rf["Feature"], top_rf["Gini_Importance"], color="#1f77b4", edgecolor="#08519c", alpha=0.85)
ax2.set_xlabel("Gini Importance (Mean Decrease in Impurity, MDI)", fontsize=10)
ax2.set_title("Top Non-linear Tree Predictors (Random Forest)", fontsize=11, fontweight="bold")
ax2.grid(axis="x", linestyle="--", alpha=0.6)

plt.tight_layout()
plt.savefig("results/day10_feature_coefficients_vs_importance.png", dpi=300)
plt.show()
plt.close()

## 11. Practical Trade-offs in Network Operations

When evaluating these models for practical network monitoring, several operational considerations arise:

1. **False Alarms vs. Detection Recall**:
   - *False Positives*: In network monitoring, false alarms can lead to operator fatigue or unnecessary investigation workflows. On this test set, Logistic Regression produced 9 false alarms ($95.31\%$ precision), whereas Random Forest produced 0 ($100.00\%$ precision).
   - *False Negatives*: Missed anomalies mean unflagged degradations. Logistic Regression missed 64 anomalies ($74.09\%$ recall), while Random Forest missed 17 ($93.12\%$ recall), detecting 48 additional anomalous samples on this test set.

2. **Measured Inference Latency**:
   - Logistic Regression evaluated 1,000 samples in **0.10 ms** on the test system.
   - Random Forest evaluated 1,000 samples in **8.26 ms** on the test system.
   - *Note on SLA Compliance*: No formal operational SLA was defined for this benchmark. Whether 8.26 ms per 1,000 samples is acceptable depends on the target system's ingestion interval, batch size, and deployment hardware.

3. **Model Complexity & Representation**:
   - Logistic Regression uses 65 scalar parameters (64 weights + 1 intercept).
   - Random Forest comprises 29,136 total decision and leaf nodes across 100 trees (mean depth 30.80). These represent fundamentally different structural units.

## 12. TelecomTS Limitations & Scope Boundaries

To maintain scientific rigor, we document key limitations of the TelecomTS benchmark:
1. **Window-Level Summary Statistics vs. Sequential Dynamics**:
   - The 128 time-step traces are aggregated into 4 static moments (mean, std, min, max). Temporal sequence order is not retained, which limits the model's ability to detect temporal patterns that share similar mean KPI values.
2. **Absence of Cellular Topology & Neighbor-Cell Context**:
   - TelecomTS samples are isolated session traces. The dataset does not include base station identifiers, cell neighbor lists, handover adjacency matrices, or geographic coordinates.
3. **Synthetic / Semi-Synthetic Anomaly Injection**:
   - Anomalies in TelecomTS follow synthetic mathematical injection patterns. Real cellular environments exhibit unpredictable stochastic variations and hardware degradations.
4. **Academic Benchmark Scope**:
   - TelecomTS is an open-access research benchmark developed by Ali Maatouk et al. Results reflect algorithmic performance on this benchmark dataset and do not represent measured performance or infrastructure of SLTMobitel's commercial cellular network.

## 13. Summary & Final Conclusions

- **Performance Comparison on Test Split**: On the TelecomTS test partition ($N=6,400$), the **Random Forest model** achieved higher test metrics across all dimensions: **100.00% precision, 93.12% recall, 96.44% F1-score, 99.98% ROC-AUC, and 99.54% PR-AUC** (compared to Logistic Regression: 95.31% precision, 74.09% recall, 83.37% F1, 97.85% ROC-AUC, 88.55% PR-AUC).
- **Error Breakdown**: Random Forest eliminated all 9 false positives seen in Logistic Regression and recovered 48 anomalies missed by Logistic Regression, while missing 1 anomaly (Sample 1047) that Logistic Regression identified.
- **Future Work**: Exploring sequential sequence models (e.g., recurrent or temporal architectures) to capture the 16 temporal anomalies that windowed summary statistics fail to distinguish.